# Fabric Governance
Tenant-wide inventory, stale-content detection, archival, and ownership management for Microsoft Fabric / Power BI.

**Companion to the GetActivities notebook**, which collects the daily activity log into a Lakehouse table. This notebook reads that table; it does not collect activity itself.

| Section | Type | Schedule |
|---|---|---|
| 1. Inventory | Read-only | Weekly |
| 2. Stale scoring | Read-only | Weekly |
| 3. Orphan / ownership report | Read-only | Weekly |
| 4. Archive & delete | **Writes / deletes** | Manual only |
| 5. Ownership takeover | **Writes** | Manual only |

**Prerequisites**
- Run as a **Fabric Administrator** (scheduled runs execute as the notebook owner).
- Attach a **default Lakehouse** (same one GetActivities writes to).
- Sections 4 and 5 only execute when `RUN_ACTIONS = True` **and** item IDs are explicitly listed. Nothing is deleted or taken over automatically.

## Parameters

In [ ]:
# ---- Activity source (written by the GetActivities notebook) ----
ACTIVITY_TABLE   = "activity_events"   # change to match your GetActivities output table

# ---- Stale rule ----
STALE_DAYS       = 90                  # no use AND no modification within this many days
INCLUDE_PERSONAL = False               # include "My workspace" content in stale scoring

# ---- Output tables ----
T_INVENTORY  = "gov_item_inventory"
T_WORKSPACES = "gov_workspace_inventory"
T_STALE      = "gov_stale_candidates"
T_ORPHANS    = "gov_orphan_workspaces"
T_ACTION_LOG = "gov_action_log"

# ---- Actions (sections 4 & 5) - leave False for scheduled runs ----
RUN_ACTIONS          = False
ARCHIVE_ITEMS        = []   # list of {"workspaceId": "...", "itemId": "..."} approved for archive
DELETE_AFTER_ARCHIVE = False
TAKEOVER_DATASETS    = []   # list of {"workspaceId": "...", "datasetId": "..."} - caller becomes owner
TAKEOVER_PAGINATED   = []   # list of {"workspaceId": "...", "reportId": "..."}
ADD_WORKSPACE_ADMINS = []   # list of {"workspaceId": "...", "email": "new.owner@contoso.com"}
ARCHIVE_ROOT         = "/lakehouse/default/Files/archive"

## 0. Setup

In [ ]:
import sempy.fabric as fabric
import pandas as pd, time, json, base64, os
from datetime import datetime, timezone

fab = fabric.FabricRestClient()     # https://api.fabric.microsoft.com/
pbi = fabric.PowerBIRestClient()    # https://api.powerbi.com/

FAB_BASE = "https://api.fabric.microsoft.com/"
NOW      = pd.Timestamp.now(tz="UTC")
CUTOFF   = NOW - pd.Timedelta(days=STALE_DAYS)

def _get(client, url, retries=5):
    """GET with basic 429 / 5xx back-off."""
    for i in range(retries):
        r = client.get(url)
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(int(r.headers.get("Retry-After", 2 ** (i + 2)))); continue
        r.raise_for_status(); return r
    r.raise_for_status(); return r

def fab_get_all(path, key):
    """Page through a Fabric admin list API using continuationToken."""
    rows, url = [], path
    while url:
        j = _get(fab, url).json()
        rows += j.get(key, [])
        tok = j.get("continuationToken")
        url = f"{path}{'&' if '?' in path else '?'}continuationToken={tok}" if tok else None
    return rows

def save(df, name, mode="overwrite"):
    """Write a pandas frame to a Delta table; objects -> string, keep datetimes."""
    out = df.copy()
    for c in out.columns:
        if not pd.api.types.is_datetime64_any_dtype(out[c]) and not pd.api.types.is_numeric_dtype(out[c]):
            out[c] = out[c].astype(str)
    out.columns = [c.replace(".", "_") for c in out.columns]
    spark.createDataFrame(out).write.mode(mode).option("overwriteSchema", "true").saveAsTable(name)
    print(f"{name}: {len(out):,} rows")

def log_action(action, ws_id, item_id, status, detail=""):
    save(pd.DataFrame([{"timestampUtc": pd.Timestamp.now(tz="UTC"), "action": action,
                        "workspaceId": ws_id, "itemId": item_id, "status": status,
                        "detail": str(detail)[:4000]}]), T_ACTION_LOG, mode="append")

## 1. Bulk inventory (read-only)

In [ ]:
ws = pd.json_normalize(fab_get_all("v1/admin/workspaces", "workspaces"))
ws = ws.rename(columns={"id": "workspaceId", "name": "workspaceName",
                        "type": "workspaceType", "state": "workspaceState"})

items = pd.json_normalize(fab_get_all("v1/admin/items", "itemEntities"))
items = items.rename(columns={"id": "itemId", "name": "itemName", "type": "itemType", "state": "itemState"})

inv = items.merge(ws[["workspaceId", "workspaceName", "workspaceType", "workspaceState"]],
                  on="workspaceId", how="left")
inv["lastUpdatedDate"] = pd.to_datetime(inv.get("lastUpdatedDate"), utc=True, errors="coerce")
inv["snapshotUtc"] = NOW

save(ws,  T_WORKSPACES)
save(inv, T_INVENTORY)
display(inv.groupby("itemType").size().sort_values(ascending=False).rename("count").to_frame())

## 2. Stale content scoring (read-only)
An item is **stale** when it has had no use (view / run / refresh / edit) **and** no modification within `STALE_DAYS`.

Activity history comes from the GetActivities table. Because the activity log only retains 28 days, items with no recorded activity are only meaningful once that table has at least `STALE_DAYS` of history - the `historyDays` value below tells you how much you have. Until then, the built-in **unused artifacts** API (30-day window) is merged in as a supplemental signal.

In [ ]:
USE_ACTIVITIES = ["ViewReport", "ViewDashboard", "ViewTile", "ExportReport", "EditReport",
                  "RefreshDataset", "ReadArtifact", "RunArtifact", "ViewArtifact",
                  "UpdateArtifact", "ExecuteNotebook", "RunNotebook"]

act_cols = [c.lower() for c in spark.table(ACTIVITY_TABLE).columns]
id_cols  = [c for c in ["ArtifactId", "ItemId", "ReportId", "DatasetId", "DashboardId", "ObjectId"]
            if c.lower() in act_cols]
time_col = next(c for c in ["CreationTime", "creationTime", "EventTime"] if c.lower() in act_cols)
in_list  = ",".join(f"'{a}'" for a in USE_ACTIVITIES)

last_use = spark.sql(f"""
    SELECT lower(COALESCE({', '.join(id_cols)})) AS itemId,
           MAX(to_timestamp({time_col}))          AS lastAccessed,
           COUNT(*)                               AS useEvents
    FROM   {ACTIVITY_TABLE}
    WHERE  Activity IN ({in_list})
    GROUP  BY 1""").toPandas()

hist = spark.sql(f"SELECT MIN(to_timestamp({time_col})) AS f FROM {ACTIVITY_TABLE}").collect()[0]["f"]
history_days = (NOW - pd.Timestamp(hist, tz="UTC")).days if hist else 0
print(f"Activity history available: {history_days} days (need {STALE_DAYS} for full confidence)")

In [ ]:
# Supplemental: built-in 'unused in last 30 days' list per workspace (Power BI items)
unused_rows = []
scope = ws[ws["workspaceState"].str.lower() == "active"]
if not INCLUDE_PERSONAL:
    scope = scope[scope["workspaceType"].str.lower() != "personal"]
for wid in scope["workspaceId"]:
    r = pbi.get(f"v1.0/myorg/admin/groups/{wid}/unused")
    if r.status_code == 429:
        time.sleep(int(r.headers.get("Retry-After", 30))); r = pbi.get(f"v1.0/myorg/admin/groups/{wid}/unused")
    if r.ok:
        for a in r.json().get("unusedArtifactEntities", []):
            unused_rows.append({"itemId": str(a.get("artifactId", "")).lower(), "unused30d": True,
                                "unusedLastAccessed": a.get("lastAccessedDateTime")})
unused = pd.DataFrame(unused_rows, columns=["itemId", "unused30d", "unusedLastAccessed"]).drop_duplicates("itemId")
print(f"Unused-in-30-days items reported: {len(unused):,}")

In [ ]:
s = inv.copy()
s["itemId_l"] = s["itemId"].str.lower()
s = s.merge(last_use, left_on="itemId_l", right_on="itemId", how="left", suffixes=("", "_u")) \
     .merge(unused,   left_on="itemId_l", right_on="itemId", how="left", suffixes=("", "_x"))
s = s.drop(columns=[c for c in ["itemId_u", "itemId_x", "itemId_l"] if c in s.columns])

s["lastAccessed"] = pd.to_datetime(s["lastAccessed"], utc=True, errors="coerce")
s["unusedLastAccessed"] = pd.to_datetime(s["unusedLastAccessed"], utc=True, errors="coerce")
s["lastAccessed"] = s[["lastAccessed", "unusedLastAccessed"]].max(axis=1)
s["useEvents"]    = s["useEvents"].fillna(0).astype(int)
s["unused30d"]    = s["unused30d"].fillna(False).astype(bool)

no_recent_use = s["lastAccessed"].isna() | (s["lastAccessed"] < CUTOFF)
no_recent_mod = s["lastUpdatedDate"].isna() | (s["lastUpdatedDate"] < CUTOFF)
s["isStale"] = no_recent_use & no_recent_mod
if not INCLUDE_PERSONAL:
    s.loc[s["workspaceType"].str.lower() == "personal", "isStale"] = False

s["staleConfidence"] = "High" if history_days >= STALE_DAYS else "Low - activity history shorter than STALE_DAYS"
s["daysSinceUse"]    = (NOW - s["lastAccessed"]).dt.days
s["daysSinceUpdate"] = (NOW - s["lastUpdatedDate"]).dt.days
s["scoredUtc"]       = NOW

save(s, T_STALE)
display(s[s["isStale"]].groupby(["workspaceName", "itemType"]).size()
          .rename("staleItems").reset_index().sort_values("staleItems", ascending=False).head(50))

**Review before acting** - exclude from archive candidates:
- Semantic models still feeding active reports (check lineage in the workspace or the Scanner API).
- Scheduled pipelines / notebooks that run but are never "viewed" (they are counted as used only if `RunArtifact` / notebook run events are in your activity table).
- Items in apps, certified/promoted content, or regulatory-retention workspaces.

## 3. Orphan / ownership report (read-only)
Finds workspaces with **no user admin** and lists item creators so owners who have left can be matched against your HR / Entra list.

In [ ]:
groups, skip = [], 0
while True:
    r = _get(pbi, f"v1.0/myorg/admin/groups?$top=5000&$skip={skip}&$expand=users&$filter=type eq 'Workspace'")
    batch = r.json().get("value", []); groups += batch
    if len(batch) < 5000: break
    skip += 5000

rows = []
for g in groups:
    users  = g.get("users", []) or []
    admins = [u for u in users if u.get("groupUserAccessRight") == "Admin"]
    user_admins = [u for u in admins if u.get("principalType") == "User"]
    rows.append({"workspaceId": g["id"], "workspaceName": g.get("name"), "state": g.get("state"),
                 "adminCount": len(admins), "userAdminCount": len(user_admins),
                 "admins": "; ".join(u.get("emailAddress") or u.get("identifier", "") for u in admins),
                 "noAdmin": len(admins) == 0, "noUserAdmin": len(user_admins) == 0})
orph = pd.DataFrame(rows)
orph["reportedUtc"] = NOW
save(orph, T_ORPHANS)
display(orph[orph["noAdmin"] | orph["noUserAdmin"]])

In [ ]:
# Item creators - join this to a departed-user list (HR extract or Entra disabled accounts)
creator_cols = [c for c in inv.columns if c.startswith("creatorPrincipal")]
creators = inv.groupby(creator_cols, dropna=False).size().rename("itemsCreated").reset_index() \
              .sort_values("itemsCreated", ascending=False)
display(creators.head(100))

## 4. Archive & delete  ⚠️ manual only
For each approved item: export its definition to `ARCHIVE_ROOT/<workspaceId>/<itemId>/`, also export a PBIX for reports where download is allowed, log the result, and optionally delete.

Notes
- Not every item type supports definition export; failures are logged, not raised.
- Lakehouse / warehouse **data** is not included in a definition - copy the data separately before deleting.
- To restore, recreate the item with `POST v1/workspaces/{id}/items` using the saved parts.

In [ ]:
def _wait_lro(r):
    loc = r.headers["Location"].replace(FAB_BASE, "")
    while True:
        time.sleep(int(r.headers.get("Retry-After", 5)))
        r = fab.get(loc); st = r.json().get("status")
        if st in ("Succeeded", "Failed", "Cancelled"): break
    if st != "Succeeded": raise RuntimeError(f"Definition export {st}: {r.text}")
    return fab.get(loc + "/result")

def export_definition(ws_id, item_id):
    r = fab.post(f"v1/workspaces/{ws_id}/items/{item_id}/getDefinition")
    if r.status_code == 202: r = _wait_lro(r)
    r.raise_for_status()
    out = f"{ARCHIVE_ROOT}/{ws_id}/{item_id}"
    for p in r.json()["definition"]["parts"]:
        path = os.path.join(out, p["path"]); os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "wb") as f: f.write(base64.b64decode(p["payload"]))
    return out

def export_pbix(ws_id, report_id):
    r = pbi.get(f"v1.0/myorg/groups/{ws_id}/reports/{report_id}/Export")
    if not r.ok: return None
    path = f"{ARCHIVE_ROOT}/{ws_id}/{report_id}/report.pbix"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f: f.write(r.content)
    return path

if RUN_ACTIONS and ARCHIVE_ITEMS:
    meta = inv.set_index("itemId")
    for a in ARCHIVE_ITEMS:
        ws_id, item_id = a["workspaceId"], a["itemId"]
        item_type = meta["itemType"].get(item_id, "") if item_id in meta.index else ""
        try:
            out = export_definition(ws_id, item_id)
            if item_type == "Report": export_pbix(ws_id, item_id)
            with open(f"{out}/_item.json", "w") as f:
                json.dump(meta.loc[item_id].astype(str).to_dict() if item_id in meta.index else a, f, indent=2)
            log_action("archive", ws_id, item_id, "Succeeded", out)
            if DELETE_AFTER_ARCHIVE:
                d = fab.delete(f"v1/workspaces/{ws_id}/items/{item_id}")
                log_action("delete", ws_id, item_id, "Succeeded" if d.ok else "Failed", d.text)
        except Exception as e:
            log_action("archive", ws_id, item_id, "Failed", e)
            print(f"Failed {item_id}: {e}")
else:
    print("Skipped - set RUN_ACTIONS = True and populate ARCHIVE_ITEMS to archive.")

## 5. Ownership changes  ⚠️ manual only
- **Semantic models / paginated reports:** *take over* makes the **identity running this notebook** the owner (a person or a governance service principal). Ownership cannot be assigned to a third party via API. Re-enter data source credentials or gateway bindings afterwards.
- **Other Fabric items** (pipelines, notebooks, warehouses, etc.): use **Take over** in the item settings where the item type supports it - support varies by type.
- **Orphaned workspaces:** add a new workspace admin via the admin API.

In [ ]:
if RUN_ACTIONS:
    for d in TAKEOVER_DATASETS:
        r = pbi.post(f"v1.0/myorg/groups/{d['workspaceId']}/datasets/{d['datasetId']}/Default.TakeOver")
        log_action("takeover_dataset", d["workspaceId"], d["datasetId"], "Succeeded" if r.ok else "Failed", r.text)
    for d in TAKEOVER_PAGINATED:
        r = pbi.post(f"v1.0/myorg/groups/{d['workspaceId']}/reports/{d['reportId']}/Default.TakeOver")
        log_action("takeover_paginated", d["workspaceId"], d["reportId"], "Succeeded" if r.ok else "Failed", r.text)
    for w in ADD_WORKSPACE_ADMINS:
        r = pbi.post(f"v1.0/myorg/admin/groups/{w['workspaceId']}/users",
                     json={"emailAddress": w["email"], "groupUserAccessRight": "Admin"})
        log_action("add_workspace_admin", w["workspaceId"], w["email"], "Succeeded" if r.ok else "Failed", r.text)
    print("Ownership actions complete - see", T_ACTION_LOG)
else:
    print("Skipped - set RUN_ACTIONS = True and populate the TAKEOVER_* / ADD_WORKSPACE_ADMINS lists.")

## Summary

In [ ]:
print(f"Workspaces:          {len(ws):,}")
print(f"Items:               {len(inv):,}")
print(f"Stale candidates:    {int(s['isStale'].sum()):,}  (>{STALE_DAYS} days, confidence: {s['staleConfidence'].iloc[0] if len(s) else 'n/a'})")
print(f"Workspaces no admin: {int(orph['noAdmin'].sum()):,}")